In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# =========================
# 1. 找到最终 75 特征数据文件
# =========================

DATA_PATH = Path("stage4_final_75_features_panel.parquet")

if not DATA_PATH.exists():
    candidates = list(Path(".").rglob("stage4_final_75_features_panel.parquet"))
    if len(candidates) == 0:
        raise FileNotFoundError("找不到 stage4_final_75_features_panel.parquet，请确认文件位置。")
    DATA_PATH = candidates[0]

print("Using file:", DATA_PATH)

# =========================
# 2. 只读取 schema，不读取完整数据
# =========================

schema = pq.read_schema(DATA_PATH)
all_columns = schema.names

print("Total columns in file:", len(all_columns))

# =========================
# 3. 排除非特征列
# =========================

non_feature_cols = {
    "date",
    "ticker",
    "instrument_id",
    "is_trade_eligible",
    "sample_split",
    "target_r_on_next",
    "target_r_on_next_winsor",
}

feature_cols = [
    c for c in all_columns
    if c not in non_feature_cols
    and not c.startswith("target_")
]

print("Number of feature columns:", len(feature_cols))

# =========================
# 4. 输出所有特征名称
# =========================

feature_name_table = pd.DataFrame({
    "feature_number": range(1, len(feature_cols) + 1),
    "feature_name": feature_cols
})

display(feature_name_table)

feature_name_table.to_csv("final_75_feature_names.csv", index=False)

print("Saved: final_75_feature_names.csv")

Using file: stage4_final_75_features_panel.parquet
Total columns in file: 83
Number of feature columns: 76


,feature_number,feature_name
0,1,eligibility_reason
1,2,r_on_mean_5
2,3,on_mom_5_20
3,4,r_on_lag1
4,5,gap_vs_on_mean20
...,...,...
71,72,dtcn_change_20
72,73,market_cap_rank
73,74,dsi_change_5_cs_rank
74,75,dsi_change_5


Saved: final_75_feature_names.csv


In [2]:
import re
import pandas as pd
from pathlib import Path

# 如果你已经运行了上一段代码，这里会直接使用 feature_cols
# 如果没有，就从 csv 读取
if "feature_cols" not in globals():
    feature_name_table = pd.read_csv("final_75_feature_names.csv")
    feature_cols = feature_name_table["feature_name"].tolist()

# =========================
# 1. 判断特征类别
# =========================

def classify_feature(name):
    n = name.lower()

    if "rank" in n or "cs_" in n or "cross" in n:
        return "cross_sectional_rank"

    if n.startswith("r_on") or n.startswith("r_id") or n.startswith("r_cc") or "return" in n:
        return "return"

    if "vol" in n or "std" in n or "sigma" in n:
        return "volatility"

    if "adv" in n or "volume" in n or "liq" in n or "spread" in n or "mktcap" in n or "size" in n or "price" in n:
        return "liquidity_size"

    if "short" in n or "dsi" in n or "dtc" in n or "dtcn" in n or "ddtcn" in n or "borrow" in n or "util" in n:
        return "short_interest"

    if "earn" in n or "bmo" in n or "amc" in n or "calendar" in n or "dow" in n or "month" in n or "weekday" in n:
        return "calendar"

    return "other_or_manual_check"


# =========================
# 2. 根据特征名生成公式
# =========================

def formula_for_feature(name):
    n = name.lower()

    # ---------- cross-sectional rank ----------
    if "rank" in n:
        base = (
            name.replace("_rank", "")
                .replace("rank_", "")
                .replace("_cs_rank", "")
                .replace("cs_rank_", "")
        )
        return f"Rank_t({base}) = cross-sectional percentile rank of {base} among eligible stocks on day t."

    # ---------- overnight return ----------
    if name == "r_on_today":
        return "r_ON,i,t = Open_adj,i,t / Close_adj,i,t-1 - 1."

    m = re.match(r"r_on_lag(\d+)", name)
    if m:
        k = int(m.group(1))
        return f"r_ON,i,t-{k} = Open_adj,i,t-{k} / Close_adj,i,t-{k}-1 - 1."

    m = re.match(r"r_on_mean_(\d+)", name)
    if m:
        w = int(m.group(1))
        return f"Mean of r_ON over the most recent {w} observable days up to day t."

    # ---------- intraday return ----------
    m = re.match(r"r_id_lag(\d+)", name)
    if m:
        k = int(m.group(1))
        return f"r_ID,i,t-{k} = Close_adj,i,t-{k} / Open_adj,i,t-{k} - 1."

    m = re.match(r"r_id_mean_(\d+)_lag1", name)
    if m:
        w = int(m.group(1))
        return f"Mean of r_ID from day t-{w} to day t-1. Current-day close is excluded."

    # ---------- close-to-close return ----------
    m = re.match(r"r_cc_lag(\d+)", name)
    if m:
        k = int(m.group(1))
        return f"r_CC,i,t-{k} = Close_adj,i,t-{k} / Close_adj,i,t-{k}-1 - 1."

    m = re.match(r"r_cc_mean_(\d+)_lag1", name)
    if m:
        w = int(m.group(1))
        return f"Mean of r_CC from day t-{w} to day t-1. Current-day close is excluded."

    # ---------- volatility ----------
    if "vol" in n or "std" in n:
        return "Rolling standard deviation of past observable returns, using only data dated t-1 or earlier unless the base variable is r_ON,t, which is observable after the open on day t."

    # ---------- liquidity / size ----------
    if "adv" in n:
        return "Average daily dollar volume over a past rolling window, usually ADV_N,i,t = mean(DollarVolume_i,u) for u <= t-1."

    if "volume" in n:
        return "Past trading volume or rolling average of past volume. Should use only volume from t-1 or earlier."

    if "spread" in n:
        return "Past liquidity spread measure, computed from historical prices/returns up to t-1."

    if "mktcap" in n or "size" in n:
        return "Size proxy, usually log market capitalization based on shares outstanding and the latest observable adjusted close, normally Close_adj,t-1."

    if "price" in n:
        return "Price-level feature based on the latest observable price, normally adjusted close from t-1 or open price on t if explicitly defined."

    # ---------- short interest ----------
    if "short" in n or "dsi" in n or "dtc" in n or "dtcn" in n or "ddtcn" in n:
        return "Short-interest feature based on the latest public short-interest report available by day t, after applying the reporting/publication lag."

    if "borrow" in n:
        return "Borrow-cost feature observable from the latest available borrow-cost data before or on day t."

    # ---------- calendar / earnings ----------
    if "earn" in n or "bmo" in n or "amc" in n:
        return "Earnings-calendar feature constructed from earnings information known by 15:50 ET on day t. AMC events are shifted to the next trading day; BMO events can be used on the announcement day."

    if "dow" in n or "weekday" in n:
        return "Day-of-week dummy or encoding based only on calendar date t."

    if "month" in n:
        return "Month dummy or calendar encoding based only on calendar date t."

    # ---------- default ----------
    return "Manual check required: formula cannot be safely inferred from the column name alone."


# =========================
# 3. 生成 15:50 ET 可观测性说明
# =========================

def observability_check(name):
    group = classify_feature(name)
    n = name.lower()

    if group == "cross_sectional_rank":
        return (
            "Observable if the underlying base feature is observable at 15:50 ET on day t. "
            "The rank uses only the cross-section of eligible stocks on the same day t."
        )

    if group == "return":
        if "today" in n and "r_on" in n:
            return (
                "Observable at 15:50 ET because Open_t and Close_t-1 are already known after the market opens."
            )
        else:
            return (
                "Observable at 15:50 ET because it uses lagged returns from t-1 or earlier. "
                "Any feature involving Close_t must be lagged."
            )

    if group == "volatility":
        return (
            "Observable at 15:50 ET if the rolling window uses only returns available before 15:50 ET, "
            "normally up to t-1."
        )

    if group == "liquidity_size":
        return (
            "Observable at 15:50 ET if volume, ADV, spread, and market-cap inputs are lagged to t-1. "
            "Full-day volume on day t must not be used."
        )

    if group == "short_interest":
        return (
            "Observable at 15:50 ET only after applying the short-interest reporting/publication lag. "
            "The value used on day t must be the latest public value available by that time."
        )

    if group == "calendar":
        return (
            "Observable at 15:50 ET if based on the public earnings calendar or calendar date. "
            "AMC earnings must be shifted to t+1 because the announcement happens after the close."
        )

    return "Manual check required."


def lookahead_reason(name):
    group = classify_feature(name)

    if group == "cross_sectional_rank":
        return (
            "No lookahead if the base feature has no lookahead. Ranking across stocks on day t does not use future returns."
        )

    if group == "return":
        return (
            "No lookahead because the feature uses Open_t, Close_t-1, or returns from t-1 and earlier. "
            "It does not use Open_t+1 or the target return."
        )

    if group == "volatility":
        return (
            "No lookahead if the rolling standard deviation is computed from past returns only."
        )

    if group == "liquidity_size":
        return (
            "No lookahead if ADV, volume, spread, and size variables are lagged and do not include full-day t volume or Close_t."
        )

    if group == "short_interest":
        return (
            "No lookahead if the short-interest value is shifted by the publication lag and forward-filled only after it becomes public."
        )

    if group == "calendar":
        return (
            "No lookahead if only known/scheduled earnings information is used, and AMC announcements are shifted forward because they occur after market close."
        )

    return "Manual check required before using this feature in the report."


# =========================
# 4. 输出最终表格
# =========================

rows = []

for i, col in enumerate(feature_cols, start=1):
    rows.append({
        "feature_number": i,
        "feature_name": col,
        "feature_group": classify_feature(col),
        "formula_or_definition": formula_for_feature(col),
        "observable_at_1550_ET_on_day_t": observability_check(col),
        "why_no_lookahead_bias": lookahead_reason(col),
        "manual_check_needed": "YES" if classify_feature(col) == "other_or_manual_check" or "Manual check" in formula_for_feature(col) else "NO"
    })

feature_formula_table = pd.DataFrame(rows)

display(feature_formula_table)

feature_formula_table.to_csv("final_75_feature_formula_lookahead_table.csv", index=False)

print("Saved: final_75_feature_formula_lookahead_table.csv")

# 单独输出需要人工检查的特征
manual_check = feature_formula_table[feature_formula_table["manual_check_needed"] == "YES"]
manual_check.to_csv("features_need_manual_formula_check.csv", index=False)

print("Saved: features_need_manual_formula_check.csv")
print("Number of features needing manual check:", len(manual_check))

,feature_number,feature_name,feature_group,formula_or_definition,observable_at_1550_ET_on_day_t,why_no_lookahead_bias,manual_check_needed
0,1,eligibility_reason,other_or_manual_check,Manual check required: formula cannot be safel...,Manual check required.,Manual check required before using this featur...,YES
1,2,r_on_mean_5,return,Mean of r_ON over the most recent 5 observable...,Observable at 15:50 ET because it uses lagged ...,"No lookahead because the feature uses Open_t, ...",NO
2,3,on_mom_5_20,other_or_manual_check,Manual check required: formula cannot be safel...,Manual check required.,Manual check required before using this featur...,YES
3,4,r_on_lag1,return,"r_ON,i,t-1 = Open_adj,i,t-1 / Close_adj,i,t-1-...",Observable at 15:50 ET because it uses lagged ...,"No lookahead because the feature uses Open_t, ...",NO
4,5,gap_vs_on_mean20,other_or_manual_check,Manual check required: formula cannot be safel...,Manual check required.,Manual check required before using this featur...,YES
...,...,...,...,...,...,...,...
71,72,dtcn_change_20,short_interest,Short-interest feature based on the latest pub...,Observable at 15:50 ET only after applying the...,No lookahead if the short-interest value is sh...,NO
72,73,market_cap_rank,cross_sectional_rank,Rank_t(market_cap) = cross-sectional percentil...,Observable if the underlying base feature is o...,No lookahead if the base feature has no lookah...,NO
73,74,dsi_change_5_cs_rank,cross_sectional_rank,Rank_t(dsi_change_5_cs) = cross-sectional perc...,Observable if the underlying base feature is o...,No lookahead if the base feature has no lookah...,NO
74,75,dsi_change_5,short_interest,Short-interest feature based on the latest pub...,Observable at 15:50 ET only after applying the...,No lookahead if the short-interest value is sh...,NO


Saved: final_75_feature_formula_lookahead_table.csv
Saved: features_need_manual_formula_check.csv
Number of features needing manual check: 15


In [3]:
import json
from pathlib import Path
import pandas as pd

feature_name_table = pd.read_csv("final_75_feature_names.csv")
feature_cols = feature_name_table["feature_name"].tolist()

code_files = []
for suffix in ["*.py", "*.ipynb"]:
    code_files.extend(Path(".").rglob(suffix))

results = []

for file in code_files:
    try:
        if file.suffix == ".py":
            text = file.read_text(encoding="utf-8", errors="ignore")
            lines = text.splitlines()

        elif file.suffix == ".ipynb":
            nb = json.loads(file.read_text(encoding="utf-8", errors="ignore"))
            lines = []
            for cell in nb.get("cells", []):
                source = cell.get("source", [])
                if isinstance(source, list):
                    lines.extend(source)
                else:
                    lines.extend(source.splitlines())

        else:
            continue

        for feature in feature_cols:
            for line_no, line in enumerate(lines, start=1):
                if feature in line:
                    results.append({
                        "feature_name": feature,
                        "file": str(file),
                        "line_number": line_no,
                        "code_line": line.strip()
                    })

    except Exception as e:
        print("Could not read:", file, e)

code_evidence = pd.DataFrame(results)

display(code_evidence)

code_evidence.to_csv("feature_formula_code_evidence.csv", index=False)

print("Saved: feature_formula_code_evidence.csv")

Could not read: 75_features_explained.ipynb Expecting value: line 1 column 1 (char 0)


,feature_name,file,line_number,code_line
0,month,stage3_borrow_cost.py,17,and forward-filled between bi-monthly releases...
1,dtcn,stage3_borrow_cost.py,53,# dtcn = days-to-cover). Among eligible names...
2,dtcn,stage3_borrow_cost.py,54,"# dtcn p90 ~ 9.5, p99 ~ 21.5. We therefore read:"
3,dtcn,stage3_borrow_cost.py,56,# or dtcn >= 10 (>= ~2 trading weeks to co...
4,dtcn,stage3_borrow_cost.py,57,# * dsi >= 0.20 or dtcn >= 20 (roughly the ...
...,...,...,...,...
2362,market_cap_rank,Step_4_feature_selection.ipynb,285,"""market_cap_rank_cs_z"","
2363,market_cap_rank,Step_4_feature_selection.ipynb,378,"""market_cap_rank_cs_rank"","
2364,market_cap_rank,Step_4_feature_selection.ipynb,379,"""market_cap_rank_cs_z"","
2365,dsi_change_5,Step_4_feature_selection.ipynb,292,"""dsi_change_5_cs_z"","


Saved: feature_formula_code_evidence.csv
